# 05 — Top-down de punta a punta: preprocesamiento, modelado y submit

Un solo notebook autocontenido que implementa la metodología top-down y deja una
entrega en Kaggle. No depende del pipe de `src/pipe/`: lee los crudos y hace todo.

## La descomposición

```
tn(producto, t+2)  =  Total(cat3, t+2)  ×  share(producto, t+2)
```

En vez de predecir las toneladas del producto directo, se predicen **dos cosas más
fáciles** y se multiplican:

| Componente | Por qué es más fácil | Cómo se predice acá |
|---|---|---|
| `Total(cat3)` | el ruido idiosincrático de cada producto se cancela al sumar | media móvil de 3 meses **o** LightGBM — se elige en validación |
| `share(producto)` | revierte a la media: es una participación, no un nivel | LightGBM con features de ciclo de vida y competencia |

Viene de `03_Techo_categoria_y_shares` (que testeó las hipótesis H1–H4) y de
`05_TopDown_share_modelado` (que hizo el test justo, con el share modelado).

## Tres cosas que hacen que el test sea honesto

1. **Optuna minimiza el WAPE en toneladas del producto**, no el error del share.
   Un share bien predicho sobre un total mal predicho no sirve de nada.
2. **Restricción de *adding-up***: los shares predichos de cada categoría se
   renormalizan para sumar 1. El total queda anclado al modelo del agregado y los
   shares sólo deciden el reparto.
3. **Se entrena un bottom-up de referencia** — mismo algoritmo, mismas features,
   misma semilla — que predice `tn` directo. Es la única forma de saber si la
   descomposición aporta algo o es sofisticación decorativa.

## Nivel de agregación

La descomposición es **producto-mes**: `share` es la participación del producto en su
categoría, y la entrega de Kaggle es por producto. Eso hace que todo el notebook
corra con **todos** los productos en minutos (~44.000 filas), sin los problemas de
memoria del pipe producto-cliente.

Meter la dimensión cliente pediría una jerarquía de tres niveles
(`cat3 → producto → cliente`) con dos renormalizaciones encadenadas. Es una extensión
válida pero no hace falta para el submit.

## 0 — Ambiente

In [ ]:
import json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
RUTA_EXP = BUCKET / "exp_topdown"
RUTA_EXP.mkdir(parents=True, exist_ok=True)

print(f"BUCKET : {BUCKET}")
print(f"crudos : {DIR_RAW}")
print(f"salida : {RUTA_EXP}")

## 1 — Palancas

In [ ]:
def rango_meses(desde: int, hasta: int) -> list:
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


PARAM = {
    # ── El horizonte. La fila de t predice t+HORIZONTE. Define el gap obligatorio
    # entre train, validacion y test: sin el, el target de las ultimas filas de train
    # ES el periodo que se esta validando.
    'horizonte': 2,

    # ── NIVEL DEL AGREGADO ───────────────────────────────────────────────
    # Dentro de que grupo se calcula el share. Es LA palanca de esta metodologia:
    #   'cat3'    -> el mas fino (99 categorias). Shares mas grandes y estables, pero
    #                el total de cada grupo es mas ruidoso (menos productos que sumar).
    #   'cat2'    -> intermedio
    #   'cat1'    -> grueso: totales muy estables, shares chiquitos y volatiles
    #   'brand'   -> por marca, en vez de por jerarquia de producto
    #   'mercado' -> un unico total. Maxima estabilidad del agregado, shares minimos.
    # El trade-off: agregar mas hace el total mas facil y el share mas dificil.
    'nivel_agregado': 'cat3',

    # ── Particion temporal (mismos defaults que 03_Optuna del pipe) ──────
    'meses_train': rango_meses(201701, 201905),
    'meses_val':   [201907, 201908],
    'meses_test':  [201910],
    'reentrenar_con_val_para_test': True,

    # ── Densificacion ────────────────────────────────────────────────────
    # 'full' -> todos los productos en todos los meses, con 0 donde no hubo venta.
    #           Es lo que garantiza que los 780 productos a entregar tengan fila en
    #           201912 para poder predecir 202002, incluso los que dejaron de venderse.
    #           Y ademas le ensenia al modelo que un producto muerto sigue muerto.
    # 'life' -> cada producto existe solo entre su primera y su ultima venta. Mas
    #           limpio conceptualmente, pero deja sin prediccion a los discontinuados.
    'densificar': 'full',

    # ── Features ─────────────────────────────────────────────────────────
    'max_lags': 12,          # lags y medias moviles de tn / total / share
    'meses_nuevo': 6,        # hasta que edad un producto cuenta como 'nuevo'

    # ── Modelo del total del agregado ────────────────────────────────────
    # 'auto' -> se prueban ma3 y lgbm y se elige el de menor WAPE en VALIDACION.
    # Forzarlo a 'ma3' o 'lgbm' sirve para comparar en el leaderboard.
    'modelo_total': 'auto',

    # ── Optuna sobre el modelo del SHARE ─────────────────────────────────
    # OJO: minimiza el WAPE en TONELADAS del producto despues de reconstruir, no el
    # error del share. Un share perfecto sobre un total malo no sirve.
    'n_trials': 40,
    'techo_arboles': 500,

    # ── Entrega ──────────────────────────────────────────────────────────
    'periodo_objetivo': 202002,
    'semillas_ensemble': [102191],
    'clip_min': 0.0,
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit': False,          # True = sube el CSV con la API de kaggle
    'mensaje_submit': None,

    'semilla': 102191,
    'sufijo': '',
}

H = PARAM['horizonte']
NIVEL = PARAM['nivel_agregado']

EXPERIMENTO = (f"topdown_{NIVEL}_{PARAM['densificar']}_{PARAM['max_lags']}lags"
               f"_total-{PARAM['modelo_total']}"
               f"_val{PARAM['meses_val'][0]}-{PARAM['meses_val'][-1]}"
               f"_test{PARAM['meses_test'][0]}"
               + (f"_{PARAM['sufijo']}" if PARAM['sufijo'] else ""))
DIR_OUT = RUTA_EXP / EXPERIMENTO
DIR_OUT.mkdir(parents=True, exist_ok=True)

print(f"EXPERIMENTO : {EXPERIMENTO}")
print(f"carpeta     : {DIR_OUT.relative_to(BUCKET)}")
print(f"\nnivel del agregado: {NIVEL}   horizonte: {H}")

## 2 — Preprocesamiento: el panel producto-mes

Una fila por (producto, mes). El `tn` se suma sobre todos los clientes — que es
exactamente la agregación con la que evalúa Kaggle.

In [ ]:
sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
          .unique(subset=["product_id"]))

print(f"sell-in  : {sell.height:,} filas · {sell['product_id'].n_unique()} productos "
      f"· {sell['customer_id'].n_unique()} clientes")


def a_m(periodo):
    """AAAAMM -> indice de mes continuo, para poder sumar y restar meses."""
    return (periodo // 100) * 12 + (periodo % 100)


def m_a_periodo(m):
    return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1


# Colapsar la dimension cliente: es la agregacion que mide la competencia.
panel = (sell.group_by(["product_id", "periodo"])
             .agg(pl.col("tn").sum().alias("tn"),
                  pl.col("cust_request_tn").sum().alias("req_tn"),
                  pl.col("cust_request_qty").sum().alias("req_qty"),
                  pl.col("customer_id").n_unique().alias("n_clientes"),
                  pl.col("plan_precios_cuidados").max().alias("precios_cuidados"))
             .with_columns((a_m(pl.col("periodo"))).alias("m")))

M_MIN, M_MAX = panel["m"].min(), panel["m"].max()
print(f"panel    : {panel.height:,} filas · rango {m_a_periodo(M_MIN)} -> {m_a_periodo(M_MAX)}")

# ── Densificacion ────────────────────────────────────────────────────────
vida = panel.group_by("product_id").agg(
    pl.col("m").min().alias("m_nace"), pl.col("m").max().alias("m_muere"))

if PARAM['densificar'] == 'full':
    # Malla completa: todos los productos en todos los meses del rango global.
    grilla = (prod.select("product_id")
                  .join(pl.DataFrame({"m": list(range(M_MIN, M_MAX + 1))}), how="cross"))
else:
    grilla = (vida.with_columns(
                    pl.int_ranges("m_nace", pl.col("m_muere") + 1).alias("m"))
                  .explode("m").select("product_id", "m"))

panel = (grilla.join(panel.drop("periodo"), on=["product_id", "m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0),
                             pl.col("req_tn").fill_null(0.0),
                             pl.col("req_qty").fill_null(0),
                             pl.col("n_clientes").fill_null(0),
                             pl.col("precios_cuidados").fill_null(0))
               .join(vida, on="product_id", how="left")
               .join(prod.select("product_id", "cat1", "cat2", "cat3", "brand", "sku_size"),
                     on="product_id", how="left")
               .with_columns(pl.col("m").map_elements(m_a_periodo, return_dtype=pl.Int64)
                               .alias("periodo")))

# El grupo dentro del cual se calcula el share.
if NIVEL == 'mercado':
    panel = panel.with_columns(pl.lit("MERCADO").alias("grupo"))
else:
    panel = panel.with_columns(pl.col(NIVEL).cast(pl.Utf8).fill_null("NA").alias("grupo"))

panel = panel.sort(["product_id", "m"])

print(f"densificado ({PARAM['densificar']}): {panel.height:,} filas · "
      f"{panel['product_id'].n_unique()} productos · {panel['grupo'].n_unique()} grupos")
print(f"ceros de tn: {(panel['tn'] == 0).sum():,} "
      f"({100*(panel['tn'] == 0).sum()/panel.height:.0f}%)")

## 3 — Features causales

Todas se calculan con información disponible **hasta el mes de la fila**. Nada de
máximos globales, largo de vida total ni fechas de muerte — ése fue el leakage
silencioso que marcó `01_EDA_series`.

La **fase** se calcula de forma expansiva: el pico es el máximo *hasta ese mes*, no
el de toda la serie.

In [ ]:
L = PARAM['max_lags']

# ── Total del agregado por (grupo, mes) ──────────────────────────────────
tot = (panel.group_by(["grupo", "m"])
            .agg(pl.col("tn").sum().alias("tn_grupo"),
                 pl.len().alias("n_prod_grupo"))
            .sort(["grupo", "m"]))

tot = tot.with_columns(
    *[pl.col("tn_grupo").shift(k).over("grupo").alias(f"tn_grupo_lag{k}")
      for k in range(1, L + 1)],
    *[pl.col("tn_grupo").rolling_mean(w).over("grupo").alias(f"tn_grupo_ma{w}")
      for w in (3, 6, 12)],
    # estacionalidad del agregado
    (((pl.col("m") - 1) % 12) + 1).alias("mes_del_anio_grupo"),
    # target del agregado: el total del grupo H meses despues
    pl.col("tn_grupo").shift(-H).over("grupo").alias("y_tn_grupo"),
)

# ── Panel del producto ───────────────────────────────────────────────────
df = panel.join(tot.select("grupo", "m", "tn_grupo", "n_prod_grupo"),
                on=["grupo", "m"], how="left")

# share: la participacion del producto en su grupo, ese mes
df = df.with_columns(
    pl.when(pl.col("tn_grupo").abs() > 1e-9)
      .then(pl.col("tn") / pl.col("tn_grupo"))
      .otherwise(0.0).alias("share"))

df = df.sort(["product_id", "m"]).with_columns(
    # historia propia
    *[pl.col("tn").shift(k).over("product_id").alias(f"tn_lag{k}") for k in range(1, L + 1)],
    *[pl.col("tn").rolling_mean(w).over("product_id").alias(f"tn_ma{w}") for w in (3, 6, 12)],
    *[pl.col("share").shift(k).over("product_id").alias(f"share_lag{k}") for k in range(1, L + 1)],
    *[pl.col("share").rolling_mean(w).over("product_id").alias(f"share_ma{w}") for w in (3, 6, 12)],
    pl.col("req_tn").rolling_mean(3).over("product_id").alias("req_tn_ma3"),
    pl.col("n_clientes").rolling_mean(3).over("product_id").alias("n_clientes_ma3"),
    # pico expansivo: el maximo HASTA este mes, no el de toda la serie
    pl.col("tn").cum_max().over("product_id").alias("tn_pico_hasta_aca"),
    pl.col("tn").cum_sum().over("product_id").alias("tn_acum"),
    # edad causal: -1 mientras el producto todavia no vendio nunca. Con
    # densificar='full' hay filas anteriores al lanzamiento, y poner ahi
    # m - m_nace (negativo) seria saber en t que el producto se lanza en t+5.
    pl.when(pl.col("m") >= pl.col("m_nace"))
      .then(pl.col("m") - pl.col("m_nace"))
      .otherwise(-1).alias("edad"),
    ((pl.col("periodo") % 100)).alias("mes_del_anio"),
)

df = df.with_columns(
    # desvio del share respecto de su propia media movil: el "error" que hay que predecir
    (pl.col("share") - pl.col("share_ma3")).alias("share_desvio_vs_ma3"),
    (pl.col("share") - pl.col("share_lag1")).alias("share_mom"),
    (pl.col("tn") - pl.col("tn_ma3")).alias("tn_desvio_vs_ma3"),
    # que fraccion de su pico historico esta vendiendo hoy -> fase del ciclo
    pl.when(pl.col("tn_pico_hasta_aca") > 1e-9)
      .then(pl.col("tn") / pl.col("tn_pico_hasta_aca"))
      .otherwise(0.0).alias("frac_del_pico"),
    (pl.col("edad").is_between(0, PARAM['meses_nuevo'])).cast(pl.Int8).alias("es_nuevo"),
    (pl.col("tn") == 0).cast(pl.Int8).alias("mes_sin_venta"),
)

# fase del ciclo de vida, expansiva y por lo tanto causal
df = df.with_columns(
    pl.when(pl.col("edad") < 0).then(pl.lit("sin_lanzar"))
      .when(pl.col("edad") <= PARAM['meses_nuevo']).then(pl.lit("rampa"))
      .when(pl.col("frac_del_pico") >= 0.85).then(pl.lit("crecimiento"))
      .when(pl.col("frac_del_pico") >= 0.50).then(pl.lit("meseta"))
      .when(pl.col("frac_del_pico") > 0.0).then(pl.lit("declive"))
      .otherwise(pl.lit("dormido")).alias("fase"))

# ── Competencia: cuantos productos nuevos entraron al grupo hace poco ────
nacimientos = (df.filter(pl.col("m") == pl.col("m_nace"))
                 .group_by(["grupo", "m"]).agg(pl.len().alias("entradas")))
ventana = (tot.select("grupo", "m")
              .join(nacimientos, on=["grupo", "m"], how="left")
              .with_columns(pl.col("entradas").fill_null(0))
              .sort(["grupo", "m"])
              .with_columns(pl.col("entradas").rolling_sum(6).over("grupo").alias("entradas_6m")))

df = (df.join(ventana.select("grupo", "m", "entradas_6m"), on=["grupo", "m"], how="left")
        .join(tot.select(["grupo", "m"] + [c for c in tot.columns
                                           if c.startswith("tn_grupo_")]),
              on=["grupo", "m"], how="left"))

# ── Los targets ──────────────────────────────────────────────────────────
# La fila de t predice t+H. shift(-H) sobre el panel densificado y ordenado por m es
# exactamente el valor H meses despues; en el borde final queda null -> esas son las
# filas de inferencia.
df = df.sort(["product_id", "m"]).with_columns(
    pl.col("share").shift(-H).over("product_id").alias("y_share"),
    pl.col("tn").shift(-H).over("product_id").alias("y_tn"),
    # aritmetico, NO shift: con shift(-H) queda null justo en las filas de
    # inferencia, que son las que necesitan saber a que mes apuntan.
    (pl.col("m") + H).map_elements(m_a_periodo, return_dtype=pl.Int64)
      .alias("periodo_objetivo"),
)
df = df.join(tot.select("grupo", "m", "y_tn_grupo"), on=["grupo", "m"], how="left")

FEATURES = [c for c in df.columns if c not in {
    "product_id", "periodo", "m", "grupo", "m_nace", "m_muere",
    "y_share", "y_tn", "y_tn_grupo", "periodo_objetivo",
    "cat1", "cat2", "cat3", "brand",           # se agregan como categoricas aparte
}]
CAT_FEATURES = ["cat1", "cat2", "cat3", "brand", "fase"]
FEATURES = [c for c in FEATURES if c not in CAT_FEATURES] + CAT_FEATURES

df = df.with_columns([pl.col(c).cast(pl.Utf8).fill_null("NA").cast(pl.Categorical)
                      for c in CAT_FEATURES])

print(f"panel de features: {df.height:,} filas x {len(FEATURES)} features")
print(f"categoricas: {CAT_FEATURES}")
print(f"\nfases:")
print(df.group_by("fase").len().sort("len", descending=True))

## 4 — Partición temporal y control de leakage

Los meses de train / validación / test se declaran a mano en `PARAM`; acá se
**verifican**. Si algo no cierra, el notebook corta antes de entrenar.

In [ ]:
MESES_TRAIN = sorted(PARAM['meses_train'])
MESES_VAL   = sorted(PARAM['meses_val'])
MESES_TEST  = sorted(PARAM['meses_test'])

sup = df.filter(pl.col("y_share").is_not_null() & pl.col("y_tn").is_not_null())
periodos_sup = sorted(sup["periodo"].unique().to_list())

MESES_TRAIN = [m for m in MESES_TRAIN if m in periodos_sup]
MESES_VAL   = [m for m in MESES_VAL   if m in periodos_sup]
MESES_TEST  = [m for m in MESES_TEST  if m in periodos_sup]

# Las filas de inferencia: las de los ultimos H meses, sin target porque cae mas alla
# del ultimo dato. De aca sale la entrega.
MESES_INFER = sorted(df.filter(pl.col("y_tn").is_null())["periodo"].unique().to_list())[-H:]
infer = df.filter(pl.col("periodo").is_in(MESES_INFER))

errores = []


def chk(ok, msg):
    print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
    if not ok:
        errores.append(msg)


print("CONTROL DE LEAKAGE")
print("=" * 72)

for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, "train", "val"),
                     (MESES_VAL, MESES_TEST, "val", "test")):
    gap = a_m(min(b)) - a_m(max(a))
    chk(gap >= H, f"gap {na}({max(a)}) -> {nb}({min(b)}) = {gap} mes(es) >= horizonte {H}")

if PARAM['reentrenar_con_val_para_test']:
    gap_tv = a_m(min(MESES_TEST)) - a_m(max(MESES_TRAIN + MESES_VAL))
    chk(gap_tv >= H, f"gap (train+val) -> test = {gap_tv} >= {H}")

chk(not (set(MESES_TRAIN) & set(MESES_VAL)), "train y val son disjuntos")
chk(not (set(MESES_VAL) & set(MESES_TEST)), "val y test son disjuntos")
chk(max(MESES_TRAIN) < min(MESES_VAL) < max(MESES_VAL) < min(MESES_TEST),
    "orden cronologico train < val < test")
chk(not (set(MESES_INFER) & set(MESES_TRAIN + MESES_VAL + MESES_TEST)),
    f"los meses de inferencia {MESES_INFER} no se usan para entrenar/validar/testear")

# Ninguna feature puede ser el target ni derivarse de el
prohibidas = {"y_share", "y_tn", "y_tn_grupo", "periodo_objetivo"}
chk(not (set(FEATURES) & prohibidas), "ningun target esta entre las features")

# Correlacion casi perfecta feature <-> target
y_chk = sup["y_tn"].to_numpy().astype(np.float64)
sospechosas = []
for c in FEATURES:
    if c in CAT_FEATURES:
        continue
    x = sup[c].to_numpy().astype(np.float64)
    ok = np.isfinite(x) & np.isfinite(y_chk)
    if ok.sum() < 100 or x[ok].std() == 0:
        continue
    r = float(np.corrcoef(x[ok], y_chk[ok])[0, 1])
    if abs(r) > 0.999:
        sospechosas.append((c, round(r, 5)))
chk(not sospechosas, f"ninguna feature correlaciona >0.999 con y_tn  {sospechosas}")

print("=" * 72)
print(f"TRAIN {len(MESES_TRAIN)} meses: {MESES_TRAIN[0]}..{MESES_TRAIN[-1]}"
      f"   ({sup.filter(pl.col('periodo').is_in(MESES_TRAIN)).height:,} filas)")
print(f"VAL   {len(MESES_VAL)} meses: {MESES_VAL}"
      f"   ({sup.filter(pl.col('periodo').is_in(MESES_VAL)).height:,} filas)")
print(f"TEST  {len(MESES_TEST)} meses: {MESES_TEST}"
      f"   ({sup.filter(pl.col('periodo').is_in(MESES_TEST)).height:,} filas)")
print(f"INFER {len(MESES_INFER)} meses: {MESES_INFER}   ({infer.height:,} filas)"
      f"  -> objetivo {sorted(infer['periodo_objetivo'].unique().to_list())}")

if errores:
    raise RuntimeError(f"Control de leakage FALLIDO: {errores}")
print("\nControl superado.")

## 5 — La métrica y la reconstrucción

`WAPE` es idéntico al del pipe, para que los números sean comparables en el
leaderboard. Acá cada fila ya **es** un producto, así que la agregación por producto
sólo suma los meses de la ventana de evaluación.

La reconstrucción es donde vive la restricción de *adding-up*: los shares predichos
de cada grupo se recortan a 0 y se **renormalizan para sumar 1**, de modo que el
total queda anclado al modelo del agregado y los shares sólo deciden el reparto.

In [ ]:
def wape(y_real, y_pred, product_ids=None, por_producto=True) -> float:
    """WAPE en toneladas. Identico al de 03_Optuna del pipe."""
    yr = np.asarray(y_real, dtype=np.float64)
    yp = np.maximum(np.asarray(y_pred, dtype=np.float64), 0.0)
    if por_producto and product_ids is not None:
        _, inv = np.unique(np.asarray(product_ids), return_inverse=True)
        yr = np.bincount(inv, weights=yr)
        yp = np.bincount(inv, weights=yp)
    den = np.abs(yr).sum()
    return float("nan") if den == 0 else float(np.abs(yr - yp).sum() / den)


def reconstruir(bloque: pl.DataFrame, share_pred, total_pred) -> np.ndarray:
    """tn_predicho = total_del_grupo_predicho x share_renormalizado.

    `bloque` trae grupo y m (una fila por producto-mes). `total_pred` es el total
    predicho para el grupo de esa fila. La renormalizacion divide cada share por la
    suma de los shares predichos de su (grupo, mes): sin eso los shares no suman 1 y
    el total reconstruido no coincide con el que predijo el modelo del agregado.
    """
    aux = bloque.select("grupo", "m").with_columns(
        pl.Series("s", np.maximum(np.asarray(share_pred, dtype=np.float64), 0.0)),
        pl.Series("tot", np.asarray(total_pred, dtype=np.float64)))
    aux = aux.with_columns(pl.col("s").sum().over(["grupo", "m"]).alias("s_suma"))
    aux = aux.with_columns(
        pl.when(pl.col("s_suma") > 1e-12)
          .then(pl.col("s") / pl.col("s_suma"))
          .otherwise(0.0).alias("s_norm"))
    return (aux["s_norm"] * aux["tot"]).to_numpy()


def wape_de(bloque: pl.DataFrame, pred_tn) -> float:
    return wape(bloque["y_tn"].to_numpy(), pred_tn, bloque["product_id"].to_numpy())


def bloques(meses):
    return sup.filter(pl.col("periodo").is_in(meses))


tr, va, te = bloques(MESES_TRAIN), bloques(MESES_VAL), bloques(MESES_TEST)
print(f"train {tr.height:,} · val {va.height:,} · test {te.height:,} filas")

## 6 — El modelo del total del agregado

Dos candidatos y se elige **en validación**:

- **`ma3`** — la media móvil de 3 meses del total. En un agregado estable es
  sorprendentemente difícil de batir, y no tiene un solo parámetro que ajustar.
- **`lgbm`** — un LightGBM sobre el panel de grupos, con los lags y medias móviles
  del total.

In [ ]:
FEAT_TOT = [c for c in tot.columns if c not in {"grupo", "m", "y_tn_grupo"}]
# tot_all tiene TODOS los meses, incluidos los de inferencia (y_tn_grupo null):
# de ahi se predice el total del mes objetivo. tot_sup es solo para entrenar.
tot_all = tot.with_columns(
    pl.col("m").map_elements(m_a_periodo, return_dtype=pl.Int64).alias("periodo"))
tot_sup = tot_all.filter(pl.col("y_tn_grupo").is_not_null())


def tot_bloque(meses):
    return tot_sup.filter(pl.col("periodo").is_in(meses))


PARAMS_TOT = {'objective': 'regression_l1', 'metric': 'mae', 'verbosity': -1,
              'n_estimators': 400, 'learning_rate': 0.05, 'num_leaves': 31,
              'min_child_samples': 20, 'seed': PARAM['semilla'], 'n_jobs': -1,
              'deterministic': True, 'force_row_wise': True}


def entrenar_total(meses):
    b = tot_bloque(meses).to_pandas()
    m = lgb.LGBMRegressor(**PARAMS_TOT)
    m.fit(b[FEAT_TOT], b["y_tn_grupo"])
    return m


def predecir_total(modelo_lgbm, bloque_tot, metodo):
    """Devuelve el total predicho por grupo-mes segun el metodo elegido."""
    if metodo == 'ma3':
        return bloque_tot["tn_grupo_ma3"].fill_null(0.0).to_numpy()
    return modelo_lgbm.predict(bloque_tot.to_pandas()[FEAT_TOT])


_m_tot_tr = entrenar_total(MESES_TRAIN)
_va_tot = tot_bloque(MESES_VAL)
wape_tot = {}
for metodo in ('ma3', 'lgbm'):
    p = np.maximum(predecir_total(_m_tot_tr, _va_tot, metodo), 0.0)
    wape_tot[metodo] = wape(_va_tot["y_tn_grupo"].to_numpy(), p,
                            _va_tot["grupo"].to_numpy())
    print(f"  total {metodo:5s} -> WAPE del agregado en val = {wape_tot[metodo]:.5f}")

METODO_TOTAL = (PARAM['modelo_total'] if PARAM['modelo_total'] != 'auto'
                else min(wape_tot, key=wape_tot.get))
print(f"\nmodelo del total elegido: {METODO_TOTAL}"
      + ("  (por validacion)" if PARAM['modelo_total'] == 'auto' else "  (forzado en PARAM)"))


def totales_para(bloque, meses_fit):
    """Total predicho, alineado fila a fila con `bloque` (una fila por producto-mes)."""
    mt = entrenar_total(meses_fit) if METODO_TOTAL == 'lgbm' else None
    tb = tot_all.filter(pl.col("periodo").is_in(sorted(bloque["periodo"].unique().to_list())))
    pred = np.maximum(predecir_total(mt, tb, METODO_TOTAL), 0.0)
    mapa = tb.select("grupo", "m").with_columns(pl.Series("tot_pred", pred))
    return (bloque.select("grupo", "m").join(mapa, on=["grupo", "m"], how="left")
                  ["tot_pred"].fill_null(0.0).to_numpy())

## 7 — Optuna sobre el modelo del share

**Lo que se minimiza es el WAPE en toneladas del producto**, después de reconstruir
y renormalizar. No el error del share: un share perfecto sobre un total malo no
sirve de nada, y optimizar el share aislado es optimizar la métrica equivocada.

En paralelo se corre un **bottom-up de referencia** que predice `tn` directo, con las
mismas features, el mismo algoritmo y el mismo presupuesto de trials. Es la única
forma de saber si la descomposición aporta algo.

In [ ]:
def espacio(trial):
    return {
        'objective': 'regression_l1', 'metric': 'mae', 'verbosity': -1,
        'seed': PARAM['semilla'], 'n_jobs': -1,
        'deterministic': True, 'force_row_wise': True,
        'num_leaves':        trial.suggest_int('num_leaves', 15, 255),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 5e-3, 0.3, log=True),
        'n_estimators':      trial.suggest_int('n_estimators', 100, PARAM['techo_arboles']),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 200),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq':    1,
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }


def fit_lgbm(params, meses, target):
    b = bloques(meses).to_pandas()
    m = lgb.LGBMRegressor(**params)
    m.fit(b[FEATURES], b[target], categorical_feature=CAT_FEATURES)
    return m


TOT_VAL = totales_para(va, MESES_TRAIN)


def obj_topdown(trial):
    m = fit_lgbm(espacio(trial), MESES_TRAIN, "y_share")
    s = m.predict(va.to_pandas()[FEATURES])
    return wape_de(va, reconstruir(va, s, TOT_VAL))


def obj_bottomup(trial):
    m = fit_lgbm(espacio(trial), MESES_TRAIN, "y_tn")
    return wape_de(va, m.predict(va.to_pandas()[FEATURES]))


estudios = {}
for nombre, objetivo in (("topdown", obj_topdown), ("bottomup", obj_bottomup)):
    t0 = time.time()
    st = optuna.create_study(
        direction="minimize", study_name=f"{EXPERIMENTO}__{nombre}",
        sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
        storage=f"sqlite:///{Path.home() / f'optuna_td_{nombre}.db'}",
        load_if_exists=True)
    st.optimize(objetivo, n_trials=PARAM['n_trials'])
    estudios[nombre] = st
    print(f"{nombre:9s} {len(st.trials)} trials · mejor WAPE val = {st.best_value:.5f}"
          f"  ({time.time()-t0:.0f}s)")

print(f"\nmejores hiperparametros del share:")
for k, v in estudios['topdown'].best_params.items():
    print(f"   {k:22s} {v}")

## 8 — Resultados: ¿gana el top-down?

`val` es optimista por construcción (Optuna lo minimizó). El número honesto es
`test`, que se mide **una sola vez** con los hiperparámetros ya elegidos.

Se agregan dos pisos de referencia: el **naive** (repetir el mes actual) y la
**media móvil de 3 meses**. Si un modelo no le gana a esos dos, no hay nada que
ajustar.

In [ ]:
MESES_FIT_TEST = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN

P_TD = {**estudios['topdown'].best_params, 'objective': 'regression_l1', 'metric': 'mae',
        'verbosity': -1, 'seed': PARAM['semilla'], 'n_jobs': -1, 'subsample_freq': 1,
        'deterministic': True, 'force_row_wise': True}
P_BU = {**estudios['bottomup'].best_params, 'objective': 'regression_l1', 'metric': 'mae',
        'verbosity': -1, 'seed': PARAM['semilla'], 'n_jobs': -1, 'subsample_freq': 1,
        'deterministic': True, 'force_row_wise': True}


def evaluar(bloque, meses_fit):
    """Devuelve {enfoque: pred_tn} para un bloque de evaluacion."""
    tot_pred = totales_para(bloque, meses_fit)
    bp = bloque.to_pandas()

    m_sh = fit_lgbm(P_TD, meses_fit, "y_share")
    m_bu = fit_lgbm(P_BU, meses_fit, "y_tn")

    return {
        "topdown":  reconstruir(bloque, m_sh.predict(bp[FEATURES]), tot_pred),
        "bottomup": m_bu.predict(bp[FEATURES]),
        "naive":    bloque["tn"].to_numpy(),
        "ma3":      bloque["tn_ma3"].fill_null(0.0).to_numpy(),
    }, (m_sh, m_bu)


pred_val, _ = evaluar(va, MESES_TRAIN)
pred_test, (mod_sh_test, mod_bu_test) = evaluar(te, MESES_FIT_TEST)

print(f"{'enfoque':10s} {'WAPE val':>10s} {'WAPE test':>10s}")
print("-" * 32)
METRICAS = {}
for enfoque in ("topdown", "bottomup", "ma3", "naive"):
    wv = wape_de(va, pred_val[enfoque])
    wt = wape_de(te, pred_test[enfoque])
    METRICAS[enfoque] = {"val": wv, "test": wt}
    print(f"{enfoque:10s} {wv:10.5f} {wt:10.5f}")

_td, _bu = METRICAS['topdown']['test'], METRICAS['bottomup']['test']
_na = METRICAS['naive']['test']
print(f"\ntop-down vs bottom-up en test : {100*(_bu-_td)/_bu:+.1f}%"
      f"   ({'gana top-down' if _td < _bu else 'gana bottom-up'})")
print(f"top-down vs naive en test     : {100*(_na-_td)/_na:+.1f}%")

_brecha = METRICAS['topdown']['test'] - METRICAS['topdown']['val']
print(f"\nbrecha test - val del top-down: {_brecha:+.5f}"
      + ("   <- ojo, sobreajuste a validacion" if _brecha > 0.02 else ""))

### ¿Dónde gana cada uno?

Ésta es la pregunta que decide si conviene un **modelo híbrido**. La hipótesis es que
la ventaja del top-down se concentra en los productos **nuevos**, donde la curva de
vida es informativa y la serie propia es corta; en los maduros, con 24 meses de
historia, el bottom-up tiene todo lo que necesita.

La **fase** suele ordenar la ventaja mejor que la **edad**: un producto de 30 meses
que volvió a crecer se parece más a un lanzamiento que a uno de 30 meses en meseta.

In [ ]:
det = te.select("product_id", "periodo", "fase", "edad", "y_tn").with_columns(
    pl.Series("td", pred_test["topdown"]),
    pl.Series("bu", pred_test["bottomup"]))

filas = []
for col, etiquetas in (("fase", det["fase"].unique().to_list()),):
    for v in etiquetas:
        b = det.filter(pl.col(col) == v)
        if b.height < 20:
            continue
        filas.append({"corte": f"{col}={v}", "n": b.height,
                      "tn_real": round(float(b["y_tn"].sum()), 1),
                      "wape_topdown": round(wape(b["y_tn"], b["td"], b["product_id"]), 4),
                      "wape_bottomup": round(wape(b["y_tn"], b["bu"], b["product_id"]), 4)})

for lo, hi, nombre in ((0, 6, "0-6m"), (7, 12, "7-12m"), (13, 24, "13-24m"), (25, 999, "25m+")):
    b = det.filter(pl.col("edad").is_between(lo, hi))
    if b.height < 20:
        continue
    filas.append({"corte": f"edad={nombre}", "n": b.height,
                  "tn_real": round(float(b["y_tn"].sum()), 1),
                  "wape_topdown": round(wape(b["y_tn"], b["td"], b["product_id"]), 4),
                  "wape_bottomup": round(wape(b["y_tn"], b["bu"], b["product_id"]), 4)})

donde = (pl.DataFrame(filas)
           .with_columns((pl.col("wape_bottomup") - pl.col("wape_topdown")).round(4)
                         .alias("ventaja_topdown"))
           .sort("ventaja_topdown", descending=True))
print(donde)
donde.write_csv(DIR_OUT / "donde_gana_cada_uno.csv")
print("\nventaja_topdown > 0 -> ahi conviene el top-down.")
print("Si la ventaja se concentra en pocos cortes, un modelo hibrido (top-down donde la")
print("historia es corta, bottom-up donde sobra) puede ganarle a los dos por separado.")

## 9 — Entrenamiento final y entrega

Ya no hay nada que estimar: se reentrena con **todos** los meses supervisados —
incluidos los que se habían reservado para validar y testear — y se predice el mes
objetivo. Del modelo final no hay métrica honesta: la que se reporta es la de `test`.

In [ ]:
MESES_TODOS = sorted(set(periodos_sup))
print(f"reentrenando con {len(MESES_TODOS)} meses: {MESES_TODOS[0]}..{MESES_TODOS[-1]}")

ENFOQUE = PARAM.get('entregar', 'mejor_en_test')
if ENFOQUE == 'mejor_en_test':
    ENFOQUE = min(("topdown", "bottomup"), key=lambda e: METRICAS[e]['test'])
print(f"enfoque que se entrega: {ENFOQUE}"
      f"   (WAPE test {METRICAS[ENFOQUE]['test']:.5f})")

tot_infer = totales_para(infer, MESES_TODOS)
ip = infer.to_pandas()

modelos_sh, modelos_bu = [], []
for sem in PARAM['semillas_ensemble']:
    modelos_sh.append(fit_lgbm({**P_TD, 'seed': sem}, MESES_TODOS, "y_share"))
    modelos_bu.append(fit_lgbm({**P_BU, 'seed': sem}, MESES_TODOS, "y_tn"))
    print(f"  semilla {sem} entrenada")

sh_pred = np.mean([m.predict(ip[FEATURES]) for m in modelos_sh], axis=0)
bu_pred = np.mean([m.predict(ip[FEATURES]) for m in modelos_bu], axis=0)

pred_infer = infer.select("product_id", "periodo", "periodo_objetivo", "grupo").with_columns(
    pl.Series("tn_topdown", np.maximum(reconstruir(infer, sh_pred, tot_infer), PARAM['clip_min'])),
    pl.Series("tn_bottomup", np.maximum(bu_pred, PARAM['clip_min'])),
    pl.Series("share_pred", sh_pred))
pred_infer = pred_infer.with_columns(
    pl.col(f"tn_{ENFOQUE}").alias("tn_pred"))
pred_infer.write_parquet(DIR_OUT / "predicciones_inferencia.parquet")

print(f"\npredicciones: {pred_infer.height:,} filas")
print(pred_infer.group_by("periodo", "periodo_objetivo").len().sort("periodo"))

## 10 — El CSV de la entrega

Kaggle mide a nivel `product_id`. Los productos de la lista oficial sin predicción
van con **0**; que sean muchos es una señal de alarma, no algo normal.

In [ ]:
OBJ = PARAM['periodo_objetivo']
obj = pred_infer.filter(pl.col("periodo_objetivo") == OBJ)
if obj.is_empty():
    raise RuntimeError(f"No hay predicciones para {OBJ}. Disponibles: "
                       f"{sorted(pred_infer['periodo_objetivo'].unique().to_list())}")

por_producto = obj.group_by("product_id").agg(pl.col("tn_pred").sum().alias("tn"))

oficiales = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt")
submit = (oficiales.select("product_id")
                   .join(por_producto, on="product_id", how="left"))
sin_pred = int(submit["tn"].null_count())
submit = submit.with_columns(pl.col("tn").fill_null(0.0)).sort("product_id")

print(f"mes objetivo {OBJ}: {obj.height} filas -> {por_producto.height} productos")
print(f"lista oficial: {oficiales.height} productos")
print(f"\nSubmit: {submit.height} filas")
print(f"Productos SIN prediccion (van en 0): {sin_pred}")
if sin_pred > oficiales.height * 0.05:
    print("   ATENCION: mas del 5% de la lista. Revisa la densificacion antes de subir.")
print(f"\ntn   min {submit['tn'].min():.3f}   media {submit['tn'].mean():.3f}   "
      f"max {submit['tn'].max():.3f}   suma {submit['tn'].sum():,.1f}")
print(submit.head(10))

path_submit = DIR_OUT / f"submission_{OBJ}.csv"
submit.write_csv(path_submit)
shutil.copy(path_submit, RUTA_EXP / "submission_ultima.csv")
print(f"\nGuardado: {path_submit}")

## 11 — Submit a Kaggle

Necesita `~/.kaggle/kaggle.json` con permisos `600`. Con `PARAM['submit'] = False`
sólo se genera el CSV.

In [ ]:
def kaggle_cli(args):
    try:
        r = subprocess.run(["kaggle"] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or "") + (r.stderr or "")
    except FileNotFoundError:
        return False, ("La CLI de kaggle no esta instalada.  pip install kaggle\n"
                       "El CSV ya quedo generado; se puede subir a mano.")
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube nada. El CSV ya esta generado.")
else:
    kdst = Path.home() / ".kaggle" / "kaggle.json"
    kdst.parent.mkdir(parents=True, exist_ok=True)
    if not kdst.exists():
        for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if cand.exists():
                shutil.copy(cand, kdst); kdst.chmod(0o600)
                print(f"Kaggle auth copiada de {cand}")
                break
    if not kdst.exists():
        print("Sin credenciales de Kaggle: no se sube. El CSV ya esta generado.")
    else:
        kdst.chmod(0o600)
        msg = PARAM['mensaje_submit'] or (
            f"{EXPERIMENTO} | {ENFOQUE} | wape_test={METRICAS[ENFOQUE]['test']:.5f}")
        ok, salida = kaggle_cli(["competitions", "submit",
                                 "-c", PARAM['kaggle_competition'],
                                 "-f", str(path_submit), "-m", msg])
        print(f"mensaje: {msg}\n{salida}")
        print("Submit enviado." if ok else "NO se pudo subir; el CSV esta en disco.")

## 12 — Registro y leaderboard

Una fila por experimento en `exp_topdown/leaderboard_topdown.csv`, con upsert por
nombre. Es la tabla para comparar niveles de agregación entre sí.

In [ ]:
resultado = {
    'experimento': EXPERIMENTO,
    'metodologia': 'top-down (total del agregado x share renormalizado)',
    'nivel_agregado': NIVEL,
    'densificar': PARAM['densificar'],
    'horizonte': H,
    'max_lags': PARAM['max_lags'],
    'meses_train': MESES_TRAIN, 'meses_val': MESES_VAL, 'meses_test': MESES_TEST,
    'meses_inferencia': MESES_INFER, 'periodo_objetivo': OBJ,
    'modelo_total': METODO_TOTAL,
    'wape_total_agregado_val': wape_tot,
    'metricas': METRICAS,
    'enfoque_entregado': ENFOQUE,
    'n_trials': {k: len(v.trials) for k, v in estudios.items()},
    'hiper_share': estudios['topdown'].best_params,
    'hiper_bottomup': estudios['bottomup'].best_params,
    'n_features': len(FEATURES), 'features': FEATURES, 'cat_features': CAT_FEATURES,
    'n_filas_train': int(tr.height),
    'n_productos_sin_prediccion': sin_pred,
    'tn_total_entregado': float(submit['tn'].sum()),
    'semilla': PARAM['semilla'],
}
with open(DIR_OUT / "resultado.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)

fila = {
    'experimento': EXPERIMENTO, 'nivel_agregado': NIVEL,
    'densificar': PARAM['densificar'], 'max_lags': PARAM['max_lags'],
    'modelo_total': METODO_TOTAL,
    'wape_test_topdown': round(METRICAS['topdown']['test'], 5),
    'wape_test_bottomup': round(METRICAS['bottomup']['test'], 5),
    'wape_test_ma3': round(METRICAS['ma3']['test'], 5),
    'wape_test_naive': round(METRICAS['naive']['test'], 5),
    'wape_val_topdown': round(METRICAS['topdown']['val'], 5),
    'brecha_test_val': round(_brecha, 5),
    'ventaja_topdown_pct': round(100*(_bu-_td)/_bu, 2),
    'vs_naive_pct': round(100*(_na-_td)/_na, 2),
    'enfoque_entregado': ENFOQUE,
    'sin_prediccion': sin_pred,
    'tn_entregado': round(float(submit['tn'].sum()), 1),
}
path_lb = RUTA_EXP / "leaderboard_topdown.csv"
nueva = pl.DataFrame([fila])
if path_lb.exists():
    viejo = pl.read_csv(path_lb).filter(pl.col("experimento") != EXPERIMENTO)
    nueva = pl.concat([viejo, nueva], how="diagonal_relaxed")
nueva.sort("wape_test_topdown").write_csv(path_lb)

print(f"resultado.json y leaderboard en {DIR_OUT.relative_to(BUCKET)}")
for p in sorted(DIR_OUT.iterdir()):
    print(f"  - {p.name}")
print(f"\nleaderboard_topdown.csv ({nueva.height} experimentos):")
print(nueva.select("nivel_agregado", "modelo_total", "wape_test_topdown",
                   "wape_test_bottomup", "ventaja_topdown_pct", "vs_naive_pct"))